In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import os
from sklearn.metrics import classification_report
import gc
import numpy as np

In [2]:
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
tf.config.threading.set_intra_op_parallelism_threads(4)
tf.config.threading.set_inter_op_parallelism_threads(4)

In [3]:
train_dir = "./data/train"
test_dir = "./data/test"

In [4]:
IMG_SIZE = 64
BATCH_SIZE = 32 
EPOCHS = 20

In [5]:
# Load datasets
train_ds_raw = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42
)

test_ds_raw = tf.keras.preprocessing.image_dataset_from_directory(
    test_dir,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Extract class names BEFORE applying ignore_errors
class_names = train_ds_raw.class_names
num_classes = len(class_names)
print("\nClasses found:", class_names)
print(f"Number of classes: {num_classes}")

Found 92259 files belonging to 5 classes.
Found 2682 files belonging to 5 classes.

Classes found: ['galaxy', 'nebula', 'planet', 'star', 'unknown']
Number of classes: 5


In [8]:
train_ds = train_ds.ignore_errors()
test_ds = test_ds.ignore_errors()

In [9]:
model = models.Sequential([
    layers.Rescaling(1./255, input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.Conv2D(16, (3, 3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(model.summary())

C:\Users\samriti\AppData\Local\anaconda3\Lib\site-packages\keras\src\layers\preprocessing\data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)                │ (None, 64, 64, 3)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d (Conv2D)                      │ (None, 62, 62, 16)          │             448 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 31, 31, 16)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 29, 29, 32)          │           4,640 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 14, 14, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 12, 12, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 6, 6, 64)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 2304)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 64)                  │         147,520 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 5)                   │             325 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 171,429 (669.64 KB)

 Trainable params: 171,429 (669.64 KB)

 Non-trainable params: 0 (0.00 B)

None


In [ ]:
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS
)

Epoch 1/20
   2884/Unknown 156s 53ms/step - accuracy: 0.9766 - loss: 0.0745

C:\Users\samriti\AppData\Local\anaconda3\Lib\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


2884/2884 ━━━━━━━━━━━━━━━━━━━━ 160s 54ms/step - accuracy: 0.9893 - loss: 0.0359 - val_accuracy: 0.9139 - val_loss: 0.3051
Epoch 2/20
1641/2884 ━━━━━━━━━━━━━━━━━━━━ 59s 48ms/step - accuracy: 0.9950 - loss: 0.0166 

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"\nTest Accuracy: {round(test_acc * 100, 2)}%")

In [ ]:
y_true = []
y_pred = []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    pred_classes = np.argmax(preds, axis=1)
    y_true.extend(labels.numpy())
    y_pred.extend(pred_classes)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

In [ ]:
print("Model Performance Summary:")
overall_acc = (y_true == y_pred).sum() / len(y_true)
print(f"\nOverall Accuracy: {overall_acc * 100:.2f}%")
print(f"\nTotal samples: {len(y_true)}")

for i, class_name in enumerate(class_names):
    true_mask = y_true == i
    num_samples = true_mask.sum()
    
    if num_samples > 0:
        correct = ((y_true == i) & (y_pred == i)).sum()
        accuracy = correct / num_samples * 100
        
        print(f"\n{class_name}:")
        print(f"  Samples: {num_samples}")
        print(f"  Correct: {correct}")
        print(f"  Accuracy: {accuracy:.2f}%")

gc.collect()

In [ ]:
os.makedirs("results", exist_ok=True)

with open("results/model_performance.txt", "w") as f:
    f.write("MODEL PERFORMANCE\n")
    f.write(f"Test Accuracy: {round(test_acc * 100, 2)}%\n")
    f.write(f"Test Loss: {round(test_loss, 4)}\n\n")
    
    f.write("Classes:\n")
    for i, c in enumerate(class_names):
        f.write(f"  {i}: {c}\n")
    
    f.write("\n" + "=" * 60 + "\n")
    f.write("PER-CLASS PERFORMANCE\n")
    f.write("=" * 60 + "\n\n")
    
    for i, class_name in enumerate(class_names):
        class_mask = y_true == i
        if class_mask.sum() > 0:
            correct = ((y_true == i) & (y_pred == i)).sum()
            total = class_mask.sum()
            accuracy = correct / total * 100
            f.write(f"{class_name}:\n")
            f.write(f"  Samples: {total}\n")
            f.write(f"  Correct: {correct}\n")
            f.write(f"  Accuracy: {accuracy:.2f}%\n\n")
    
    f.write("CONFIGURATION\n")
    f.write(f"Image Size: {IMG_SIZE}x{IMG_SIZE}\n")
    f.write(f"Batch Size: {BATCH_SIZE}\n")
    f.write(f"Epochs: {EPOCHS}\n")

print("Performance report saved")

In [ ]:
model.save("results/model.h5")
print("Model saved")

gc.collect()